In [1]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, LayerNormalization
from tensorflow.keras.optimizers import Adam, RMSprop
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2
import warnings
warnings.filterwarnings('ignore') 

print("Library berhasil diimport!")
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

Library berhasil diimport!
TensorFlow version: 2.10.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# Load dataset
def load_split_data(split_num):
    """Load data untuk split tertentu""" 
    X_train = pd.read_csv(f'feature-engineering/X0-train-split{split_num}-seqglo-truncate.csv')
    X_test = pd.read_csv(f'feature-engineering/X0-test-split{split_num}-seqglo-truncate.csv')
    y_train = pd.read_csv(f'feature-engineering/y0-train-split{split_num}-seqglo-truncate.csv')
    y_test = pd.read_csv(f'feature-engineering/y0-test-split{split_num}-seqglo-truncate.csv')
    
    return X_train, X_test, y_train, y_test

# Load split pertama sebagai contoh
X_train, X_test, y_train, y_test = load_split_data(1)

print(f"Shape X_train: {X_train.shape}")
print(f"Shape X_test: {X_test.shape}")
print(f"Shape y_train: {y_train.shape}")
print(f"Shape y_test: {y_test.shape}")
print(f"\nFeatures: {list(X_train.columns)}")
print(f"Unique labels: {sorted(y_train['label'].unique())}")

Shape X_train: (200256, 9)
Shape X_test: (50064, 9)
Shape y_train: (200256, 1)
Shape y_test: (50064, 1)

Features: ['gazeX', 'gazeY', 'kecepatan', 'direction', 'acceleration', 'cumulative-distance', 'displacement', 'stddev_2pop', 'linearity_index']
Unique labels: [1, 2]


In [3]:
# Fungsi untuk membuat sliding window
def create_sliding_window(data, window_size=60):
    """
    Membuat sliding window untuk data time series
    
    Args:
        data: numpy array atau pandas DataFrame
        window_size: ukuran window (default 60)
    
    Returns:
        X: array 3D dengan shape (samples, window_size, features)
        y: array 1D dengan label untuk setiap window
    """
    if isinstance(data, pd.DataFrame):
        features = data.drop('label', axis=1).values if 'label' in data.columns else data.values
        labels = data['label'].values if 'label' in data.columns else None
    else:
        features = data
        labels = None
    
    X, y = [], []
    
    for i in range(len(features) - window_size + 1):
        X.append(features[i:i + window_size])
        if labels is not None:
            # Ambil label terakhir dari window
            y.append(labels[i + window_size - 1])
    
    return np.array(X), np.array(y)

# Test fungsi sliding window
print("Testing sliding window function...")
sample_data = pd.concat([X_train.head(100), y_train.head(100)], axis=1)
X_sample, y_sample = create_sliding_window(sample_data, window_size=60)
print(f"Original data shape: {sample_data.shape}")
print(f"Windowed X shape: {X_sample.shape}")
print(f"Windowed y shape: {y_sample.shape}")

Testing sliding window function...
Original data shape: (100, 10)
Windowed X shape: (41, 60, 9)
Windowed y shape: (41,)


In [4]:
# Data preprocessing dan normalisasi yang diperbaiki
def preprocess_data_improved(X_train, X_test, y_train, y_test, window_size=60):
    """
    Preprocessing data untuk LSTM dengan normalisasi yang benar
    """
    print("Melakukan preprocessing data...")
    
    # Copy data untuk menghindari modifikasi original
    X_train_copy = X_train.copy()
    X_test_copy = X_test.copy()
    y_train_copy = y_train.copy()
    y_test_copy = y_test.copy()
    
    # Normalisasi fitur - HANYA fit pada training data
    scaler = MinMaxScaler()
    
    # Fit scaler HANYA pada data training
    X_train_scaled = scaler.fit_transform(X_train_copy)
    # Transform data testing menggunakan parameter dari training
    X_test_scaled = scaler.transform(X_test_copy)
    
    print(f"Training data - Mean: {X_train_scaled.mean():.6f}, Std: {X_train_scaled.std():.6f}")
    print(f"Testing data - Mean: {X_test_scaled.mean():.6f}, Std: {X_test_scaled.std():.6f}")
    
    # Convert back to DataFrame untuk memudahkan sliding window
    X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train_copy.columns)
    X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test_copy.columns)
    
    # Gabungkan dengan labels
    train_data = pd.concat([X_train_scaled_df, y_train_copy.reset_index(drop=True)], axis=1)
    test_data = pd.concat([X_test_scaled_df, y_test_copy.reset_index(drop=True)], axis=1)
    
    # Buat sliding window
    print(f"Membuat sliding window dengan ukuran {window_size}...")
    X_train_window, y_train_window = create_sliding_window(train_data, window_size)
    X_test_window, y_test_window = create_sliding_window(test_data, window_size)
    
    # Encode labels
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train_window)
    y_test_encoded = label_encoder.transform(y_test_window)
    
    print(f"Window shapes - X_train: {X_train_window.shape}, X_test: {X_test_window.shape}")
    print(f"Label distribution - Train: {np.bincount(y_train_encoded)}, Test: {np.bincount(y_test_encoded)}")
    
    return X_train_window, X_test_window, y_train_encoded, y_test_encoded, scaler, label_encoder

# Preprocessing data dengan fungsi yang diperbaiki
print("=== Preprocessing data dengan normalisasi yang diperbaiki ===")
X_train_proc_new, X_test_proc_new, y_train_proc_new, y_test_proc_new, scaler_new, label_encoder_new = preprocess_data_improved(
    X_train, X_test, y_train, y_test, window_size=60
)

print(f"\nHasil preprocessing:")
print(f"X_train_processed shape: {X_train_proc_new.shape}")
print(f"X_test_processed shape: {X_test_proc_new.shape}")
print(f"y_train_processed shape: {y_train_proc_new.shape}")
print(f"y_test_processed shape: {y_test_proc_new.shape}")
print(f"Number of classes: {len(label_encoder_new.classes_)}")
print(f"Classes: {label_encoder_new.classes_}")

=== Preprocessing data dengan normalisasi yang diperbaiki ===
Melakukan preprocessing data...
Training data - Mean: 0.250059, Std: 0.321135
Testing data - Mean: 0.210081, Std: 0.274094
Membuat sliding window dengan ukuran 60...
Window shapes - X_train: (200197, 60, 9), X_test: (50005, 60, 9)
Label distribution - Train: [100128 100069], Test: [24973 25032]

Hasil preprocessing:
X_train_processed shape: (200197, 60, 9)
X_test_processed shape: (50005, 60, 9)
y_train_processed shape: (200197,)
y_test_processed shape: (50005,)
Number of classes: 2
Classes: [1 2]


In [5]:
input_shape_new = (X_train_proc_new.shape[1], X_train_proc_new.shape[2])
num_classes_new = len(label_encoder_new.classes_)

In [7]:
# Membuat model LSTM yang diperbaiki untuk mengatasi overfitting
def create_improved_lstm_model(input_shape, num_classes):
    """
    Membuat model LSTM dengan regularisasi yang lebih kuat untuk mengatasi overfitting
    """
    model = Sequential([
        # Layer LSTM dengan unit lebih kecil
        LSTM(256, return_sequences=True, input_shape=input_shape,
             kernel_regularizer=l2(0.001)),
        Dropout(0.5),  # Tambahkan dropout untuk mengurangi overfitting
        LayerNormalization(),
        
        LSTM(128, return_sequences=True, 
             kernel_regularizer=l2(0.001)),
        Dropout(0.5),  # Tambahkan dropout untuk mengurangi overfitting
        LayerNormalization(),
        
        LSTM(64, return_sequences=False,
             kernel_regularizer=l2(0.001)),
        Dropout(0.5),
        LayerNormalization(),
        
        # Dense layer yang lebih efisien
        Dense(128, activation='relu', kernel_regularizer=l2(0.001)),
        Dropout(0.5),
        LayerNormalization(),
        
        Dense(num_classes, activation='softmax')
    ])
    return model


# Buat model baru dengan arsitektur yang diperbaiki
input_shape_new = (X_train_proc_new.shape[1], X_train_proc_new.shape[2])
num_classes_new = len(label_encoder_new.classes_)

model_improved = create_improved_lstm_model(input_shape_new, num_classes_new)

# Compile model dengan learning rate yang lebih kecil
model_improved.compile(
    optimizer=RMSprop(learning_rate=0.0001),  # Learning rate lebih kecil
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Print model summary
print("Model Summary (Improved):")
model_improved.summary()
print(f"\nInput shape: {input_shape_new}")
print(f"Number of classes: {num_classes_new}")
print(f"Total parameters: {model_improved.count_params():,}")

Model Summary (Improved):
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 60, 256)           272384    
                                                                 
 dropout (Dropout)           (None, 60, 256)           0         
                                                                 
 layer_normalization (LayerN  (None, 60, 256)          512       
 ormalization)                                                   
                                                                 
 lstm_1 (LSTM)               (None, 60, 128)           197120    
                                                                 
 dropout_1 (Dropout)         (None, 60, 128)           0         
                                                                 
 layer_normalization_1 (Laye  (None, 60, 128)          256       
 rNormalization)              

In [8]:
def train_improved_model_with_checkpoint(model, X_train, y_train, X_test, y_test, epochs=100, batch_size=64, checkpoint_path='best_model.h5'):
    """
    Training model LSTM dengan teknik anti-overfitting yang lebih agresif dan ModelCheckpoint
    """
    
    # Buat direktori untuk checkpoint jika belum ada
    checkpoint_dir = os.path.dirname(checkpoint_path)
    if checkpoint_dir and not os.path.exists(checkpoint_dir):
        os.makedirs(checkpoint_dir)
    
    # Callbacks yang lebih agresif
    early_stopping = EarlyStopping(
        monitor='val_accuracy',
        patience=15,  # Sedikit lebih panjang karena ada checkpoint
        restore_best_weights=True,
        verbose=1,
        min_delta=0.001,
        mode='max'
    )
    
    reduce_lr = ReduceLROnPlateau(
        monitor='val_accuracy',
        factor=0.6,  # Lebih agresif dalam mengurangi LR
        patience=5,   # Lebih cepat mengurangi LR
        min_lr=1e-9,
        verbose=1,
        mode='max'
    )
    
    # ModelCheckpoint - simpan model terbaik
    model_checkpoint = ModelCheckpoint(
        filepath=checkpoint_path,
        monitor='val_accuracy',
        save_best_only=True,  # Hanya simpan model dengan val_accuracy terbaik
        save_weights_only=False,  # Simpan seluruh model (arsitektur + weights)
        mode='max',
        verbose=1,
        save_freq='epoch'  # Cek setiap epoch
    )
    
    # Training dengan batch size yang lebih besar untuk regularisasi
    print(f"Training dengan batch size: {batch_size}")
    print(f"Model checkpoint akan disimpan di: {checkpoint_path}")
    
    history  = model.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_data=(X_test, y_test),
        callbacks=[early_stopping, reduce_lr, model_checkpoint],
        verbose=1,
        # shuffle=True  # Shuffle data setiap epoch
    )
    
    return history, checkpoint_path

# Training model yang sudah diperbaiki
print("=== Memulai training model yang diperbaiki ===")
checkpoint_path = 'model_checkpoints/best_lstm_model.h5'
history_improved, saved_checkpoint = train_improved_model_with_checkpoint(
    model_improved, 
    X_train_proc_new, y_train_proc_new, 
    X_test_proc_new, y_test_proc_new,
    epochs=100,
    batch_size=256,
    checkpoint_path=checkpoint_path
)

print(f"\nTraining selesai!")
print(f"Final training loss: {history_improved.history['loss'][-1]:.4f}")
print(f"Final validation loss: {history_improved.history['val_loss'][-1]:.4f}")
print(f"Final training accuracy: {history_improved.history['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: {history_improved.history['val_accuracy'][-1]:.4f}")

=== Memulai training model yang diperbaiki ===
Training dengan batch size: 256
Model checkpoint akan disimpan di: model_checkpoints/best_lstm_model.h5
Epoch 1/100
782/783 [============================>.] - ETA: 0s - loss: 1.1895 - accuracy: 0.5162
Epoch 1: val_accuracy improved from -inf to 0.45147, saving model to model_checkpoints\best_lstm_model.h5
783/783 [==============================] - 53s 48ms/step - loss: 1.1895 - accuracy: 0.5162 - val_loss: 2.4740 - val_accuracy: 0.4515 - lr: 1.0000e-04
Epoch 2/100
782/783 [============================>.] - ETA: 0s - loss: 0.8635 - accuracy: 0.5232
Epoch 2: val_accuracy did not improve from 0.45147
783/783 [==============================] - 35s 44ms/step - loss: 0.8635 - accuracy: 0.5232 - val_loss: 2.7045 - val_accuracy: 0.4312 - lr: 1.0000e-04
Epoch 3/100
782/783 [============================>.] - ETA: 0s - loss: 0.7647 - accuracy: 0.5272
Epoch 3: val_accuracy did not improve from 0.45147
783/783 [==============================] - 37s 47m

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

def plot_history(history):
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(history.history['loss'], label='Train')
    plt.plot(history.history['val_loss'], label='Val')
    plt.title('Loss')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history.history['accuracy'], label='Train')
    plt.plot(history.history['val_accuracy'], label='Val')
    plt.title('Accuracy')
    plt.legend()
    
    plt.show()
    
plot_history(history_improved)
